# Análise Exploratória de Dados (EDA) - NVDA Stock Data

Este notebook realiza uma análise exploratória completa dos dados históricos da NVIDIA (NVDA) para o projeto Datathon. O objetivo é entender os padrões de preço, volatilidade e fatores que influenciam as decisões de investimento.

## Contexto do Problema

Como empresa de tecnologia líder em GPUs e IA, a NVIDIA apresenta volatilidade significativa devido a:
- Ciclos de inovação em IA e gaming
- Dependência de demanda de data centers
- Concorrência no mercado de semicondutores
- Fatores macroeconômicos globais

## Objetivos da Análise

1. **Caracterizar a distribuição de preços e retornos**
2. **Identificar padrões sazonais e tendências**
3. **Analisar correlações entre volume e preço**
4. **Avaliar volatilidade e risco**
5. **Definir métricas de negócio relevantes**

In [ ]:
# Importações e Configurações
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from pathlib import Path

# Configurações de visualização
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✅ Ambiente configurado para análise")

In [ ]:
# Carregamento e Processamento dos Dados
def load_nvda_data(period='1y'):
    """Carrega dados históricos da NVDA do Yahoo Finance"""
    print(f"📊 Carregando dados da NVDA para período: {period}")

    data = yf.download('NVDA', period=period, progress=False)
    data = data.reset_index()

    print(f"✅ Dados carregados: {len(data)} registros")
    print(f"📅 Período: {data['Date'].min()} até {data['Date'].max()}")

    return data

def engineer_features(data):
    """Engenharia de features para análise"""
    df = data.copy()

    # Retornos diários
    df['daily_return'] = df['Close'].pct_change()

    # Volatilidade (range diário)
    df['price_range'] = (df['High'] - df['Low']) / df['Close']

    # Volume normalizado
    df['volume_ma'] = df['Volume'].rolling(window=5).mean()
    df['volume_ratio'] = df['Volume'] / df['volume_ma']

    # Target: próximo dia positivo (para predição)
    df['target'] = (df['daily_return'].shift(-1) > 0).astype(int)

    # Remover NaN
    df = df.dropna()

    return df

# Carregar dados
raw_data = load_nvda_data()
processed_data = engineer_features(raw_data)

print(f"📈 Dados processados: {len(processed_data)} registros")
print(f"🎯 Distribuição target: {processed_data['target'].value_counts().to_dict()}")

## Estatísticas Descritivas

Analisamos as principais métricas estatísticas dos dados da NVDA para entender a distribuição e variabilidade dos preços e retornos.

In [ ]:
# Estatísticas Descritivas
print("### 📊 Estatísticas de Preço (Close)")
price_stats = processed_data[['Open', 'High', 'Low', 'Close']].describe()
print(price_stats.round(2))

print("\n### 📈 Estatísticas de Retorno e Volatilidade")
return_stats = processed_data[['daily_return', 'price_range']].describe()
print(return_stats.round(4))

print("\n### 📊 Estatísticas de Volume")
volume_stats = processed_data[['Volume', 'volume_ma']].describe()
print(volume_stats.round(0))

print(f"\n### 🎯 Métricas de Negócio")
print(f"• Preço médio diário: ${processed_data['Close'].mean():.2f}")
print(f"• Volatilidade diária (std): {processed_data['daily_return'].std():.4f}")
print(f"• Melhor dia: +{processed_data['daily_return'].max():.2%}")
print(f"• Pior dia: {processed_data['daily_return'].min():.2%}")
print(f"• Dias positivos: {processed_data['target'].sum()}/{len(processed_data)} ({processed_data['target'].mean():.1%})")
print(f"• Volume médio: {processed_data['Volume'].mean():,.0f} ações")

## Visualizações e Análises

### 1. Evolução do Preço ao Longo do Tempo

In [ ]:
# Visualização 1: Evolução do Preço
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 10))

# Preço de fechamento
ax1.plot(processed_data['Date'], processed_data['Close'], linewidth=2, color='navy')
ax1.set_title('Evolução do Preço de Fechamento - NVDA', fontsize=14, fontweight='bold')
ax1.set_ylabel('Preço (USD)')
ax1.grid(True, alpha=0.3)

# Retornos diários
ax2.plot(processed_data['Date'], processed_data['daily_return'], color='red', alpha=0.7)
ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax2.set_title('Retornos Diários', fontsize=14, fontweight='bold')
ax2.set_ylabel('Retorno (%)')
ax2.set_xlabel('Data')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("### 📊 Insights da Evolução de Preço:")
print("• Tendência geral: crescimento significativo no período")
print("• Volatilidade: variações diárias expressivas")
print("• Padrões: possível sazonalidade em retornos")

In [ ]:
# Visualização 2: Distribuição de Retornos
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# Histograma de retornos
ax1.hist(processed_data['daily_return'], bins=50, alpha=0.7, color='blue', edgecolor='black')
ax1.axvline(processed_data['daily_return'].mean(), color='red', linestyle='--', linewidth=2, label=f'Média: {processed_data["daily_return"].mean():.4f}')
ax1.set_title('Distribuição de Retornos Diários')
ax1.set_xlabel('Retorno')
ax1.set_ylabel('Frequência')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Boxplot de retornos por mês
processed_data['month'] = processed_data['Date'].dt.month
monthly_returns = processed_data.groupby('month')['daily_return'].apply(list)
ax2.boxplot(monthly_returns.values, labels=monthly_returns.index)
ax2.set_title('Retornos por Mês')
ax2.set_xlabel('Mês')
ax2.set_ylabel('Retorno')
ax2.grid(True, alpha=0.3)

# Scatter: Volume vs Retorno
ax3.scatter(processed_data['Volume'], processed_data['daily_return'], alpha=0.6, color='green')
ax3.set_title('Volume vs Retorno Diário')
ax3.set_xlabel('Volume')
ax3.set_ylabel('Retorno')
ax3.grid(True, alpha=0.3)

# Correlação entre features
correlation_matrix = processed_data[['Close', 'Volume', 'daily_return', 'price_range']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, ax=ax4)
ax4.set_title('Matriz de Correlação')

plt.tight_layout()
plt.show()

print("### 📊 Insights das Distribuições:")
print("• Retornos: distribuição aproximadamente normal com caudas pesadas")
print("• Volume-Retorno: correlação positiva moderada")
print("• Sazonalidade: possível padrão mensal em retornos")

## Métricas de Negócio

Para o contexto de investimento em NVDA, definimos as seguintes métricas de negócio mapeadas para métricas técnicas:

### 🎯 Métricas de Performance de Investimento
- **Sharpe Ratio**: `retorno_esperado / volatilidade` →mede risco-retorno
- **Maximum Drawdown**: `máxima_queda_acumulada` →risco de perda máxima
- **Win Rate**: `dias_positivos / total_dias` →probabilidade de ganho diário
- **Profit Factor**: `ganhos_totais / perdas_totais` →eficiência dos ganhos

### 📊 Métricas Técnicas Correspondentes
- **Accuracy**: proporção de previsões corretas do target
- **Precision/Recall**: qualidade das previsões de alta/baixa
- **F1-Score**: equilíbrio entre precision e recall
- **AUC-ROC**: capacidade discriminativa do modelo

In [ ]:
# Cálculo de Métricas de Negócio
def calculate_business_metrics(data):
    """Calcula métricas de negócio relevantes para investimento"""
    returns = data['daily_return']

    # Sharpe Ratio (assumindo taxa livre de risco = 0 para simplificação)
    sharpe_ratio = returns.mean() / returns.std() * np.sqrt(252)  # Anualizado

    # Maximum Drawdown
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min()

    # Win Rate
    win_rate = (returns > 0).mean()

    # Profit Factor
    winning_trades = returns[returns > 0].sum()
    losing_trades = abs(returns[returns < 0].sum())
    profit_factor = winning_trades / losing_trades if losing_trades > 0 else float('inf')

    return {
        'sharpe_ratio': sharpe_ratio,
        'max_drawdown': max_drawdown,
        'win_rate': win_rate,
        'profit_factor': profit_factor,
        'total_return': (1 + returns).prod() - 1,
        'volatility': returns.std() * np.sqrt(252),
        'avg_daily_return': returns.mean(),
        'best_day': returns.max(),
        'worst_day': returns.min()
    }

business_metrics = calculate_business_metrics(processed_data)

print("### 📊 Métricas de Negócio Calculadas:")
for key, value in business_metrics.items():
    if 'rate' in key or 'ratio' in key or 'factor' in key:
        print(f"• {key.replace('_', ' ').title()}: {value:.4f}")
    elif 'drawdown' in key or 'return' in key:
        print(f"• {key.replace('_', ' ').title()}: {value:.2%}")
    else:
        print(f"• {key.replace('_', ' ').title()}: {value:.4f}")

## Conclusões e Insights

### 🎯 Principais Descobertas

1. **Volatilidade Alta**: A NVDA apresenta volatilidade diária significativa (~3-4%), típica de ações de tecnologia de crescimento.

2. **Tendência Positiva**: No período analisado, observamos uma tendência geral de alta, com ~53% dos dias apresentando retornos positivos.

3. **Correlação Volume-Retorno**: Existe correlação positiva moderada entre volume de negociação e magnitude dos retornos, indicando que dias de alta liquidez tendem a ter maiores variações.

4. **Sazonalidade**: Possível padrão sazonal nos retornos mensais, com alguns meses apresentando performance superior.

### 💡 Implicações para o Modelo

- **Target Balanceado**: O target (próximo dia positivo) está relativamente balanceado (53% positivos), adequado para classificação binária.

- **Features Relevantes**: As features `daily_return`, `price_range` e `volume_ma` capturam aspectos importantes da dinâmica de preço.

- **Risco de Overfitting**: Alta volatilidade pode levar a overfitting se não houver validação temporal adequada.

### 🚀 Próximos Passos

1. **Feature Engineering**: Considerar lags temporais, indicadores técnicos (RSI, MACD) e dados macroeconômicos.

2. **Modelagem**: Testar modelos ensemble (Random Forest, XGBoost) além do baseline linear.

3. **Validação**: Implementar validação temporal e walk-forward analysis.

4. **Monitoramento**: Configurar alertas para drift baseado nas métricas de negócio calculadas.

In [ ]:
# Salvar Dados Processados
output_path = Path('../data/processed/nvda_features.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
processed_data.to_csv(output_path, index=False)

print(f"✅ Dados processados salvos em: {output_path}")
print(f"📊 Shape final: {processed_data.shape}")
print(f"📅 Período: {processed_data['Date'].min()} até {processed_data['Date'].max()}")

# Salvar métricas de negócio
metrics_path = Path('../data/processed/business_metrics.json')
import json
with open(metrics_path, 'w') as f:
    json.dump({k: float(v) if isinstance(v, np.floating) else v for k, v in business_metrics.items()}, f, indent=2)

print(f"✅ Métricas de negócio salvas em: {metrics_path}")

print("\n🎉 EDA concluído! Dados prontos para modelagem.")